In [1]:
import numpy as np
import torch as t

In [3]:
with open(r"C:\Users\hrish\Projects\ner_with_finetuning\prepared_dataset.json", 'r') as file:
    data = json.load(file)

In [30]:
len(data)

1014

In [26]:
data[0][1]

{'entities': [[0, 8, 'SPORT'],
  [9, 25, 'COMPETITION'],
  [26, 33, 'ARTICLE_TYPE'],
  [34, 41, 'DATE']]}

In [6]:
from huggingface_hub import notebook_login
notebook_login()

In [8]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-v3-base",use_fast=False)

In [10]:
example = data[0][0]
tokenized_input = tokenizer(example)
tokenized_input

{'input_ids': [1, 2242, 9026, 2377, 793, 4023, 2], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1]}

In [11]:
tokens = tokenizer.convert_ids_to_tokens(tokenized_input["input_ids"])
tokens

['[CLS]',
 '▁football',
 '▁Champions',
 '▁League',
 '▁results',
 '▁tonight',
 '[SEP]']

In [19]:
label2id = {
    "O": 0,

    "B-COMP": 1, "I-COMP": 2,
    "B-CONTESTANT": 3, "I-CONTESTANT": 4,
    "B-DATE": 5, "I-DATE": 6,
    "B-SPORT": 7, "I-SPORT": 8,
    "B-ARTICLE": 9, "I-ARTICLE": 10,
}

entity_map = {
    "COMPETITION": ("B-COMP", "I-COMP"),
    "CONTESTANT": ("B-CONTESTANT", "I-CONTESTANT"),
    "DATE": ("B-DATE", "I-DATE"),
    "SPORT": ("B-SPORT", "I-SPORT"),
    "ARTICLE_TYPE": ("B-ARTICLE", "I-ARTICLE"),
}

In [20]:
def tokenize_and_align_labels(batch, tokenizer):
    texts = [example[0] for example in batch]

    encodings = tokenizer(
        texts,
        padding=True,
        truncation=True,
        return_offsets_mapping=True
    )

    all_labels = []

    for batch_idx, (_, annotation) in enumerate(batch):

        offsets = encodings["offset_mapping"][batch_idx]

        # Everything ignored by default
        labels = [-100] * len(offsets)

        # ------------------------------------------------------------------
        # First subtoken of every word gets O
        # ------------------------------------------------------------------

        previous_end = None

        for token_idx, (start, end) in enumerate(offsets):

            # Special token
            if start == end:
                continue

            is_first_subtoken = (
                previous_end is None or
                start != previous_end
            )

            if is_first_subtoken:
                labels[token_idx] = label2id["O"]

            previous_end = end

        # ------------------------------------------------------------------
        # Apply entity labels
        # ------------------------------------------------------------------

        for ent_start, ent_end, ent_type in annotation["entities"]:

            if ent_type not in entity_map:
                continue

            b_tag, i_tag = entity_map[ent_type]

            first_word_in_entity = True
            previous_end = None

            for token_idx, (tok_start, tok_end) in enumerate(offsets):

                if tok_start == tok_end:
                    continue

                overlaps = (
                    tok_start < ent_end and
                    tok_end > ent_start
                )

                if not overlaps:
                    continue

                is_first_subtoken = (
                    previous_end is None or
                    tok_start != previous_end
                )

                previous_end = tok_end

                if not is_first_subtoken:
                    continue

                if first_word_in_entity:
                    labels[token_idx] = label2id[b_tag]
                    first_word_in_entity = False
                else:
                    labels[token_idx] = label2id[i_tag]

        all_labels.append(labels)

    encodings["labels"] = all_labels
    encodings.pop("offset_mapping")

    return encodings

In [21]:
import random
random.shuffle(data)
processed = tokenize_and_align_labels(data, tokenizer)

In [42]:
processed[4]

Encoding(num_tokens=18, attributes=[ids, type_ids, tokens, offsets, attention_mask, special_tokens_mask, overflowing])

In [49]:
class NERDataset(t.utils.data.Dataset):
    def __init__(self, encodings):
        self.encodings = encodings

    def __len__(self):
        return len(self.encodings["input_ids"])

    def __getitem__(self, idx):
        return {key: t.tensor(val[idx]) for key, val in self.encodings.items()}

dataset = NERDataset(processed)
train_data = t.utils.data.Subset(dataset, range(900))
test_data  = t.utils.data.Subset(dataset, range(900, len(dataset)))

In [45]:
from transformers import DataCollatorForTokenClassification
data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

In [46]:
import evaluate
seqeval = evaluate.load("seqeval")

In [47]:
id2label = {v: k for k, v in label2id.items()}

In [48]:
import numpy as np

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    # (batch, seq_len, num_labels)
    predictions = np.argmax(predictions, axis=-1)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(predictions, labels):

        pred_labels = []
        gold_labels = []

        for pred_id, gold_id in zip(pred_seq, label_seq):

            # Ignore special tokens and continuation subtokens
            if gold_id == -100:
                continue

            pred_labels.append(id2label[pred_id])
            gold_labels.append(id2label[gold_id])

        true_predictions.append(pred_labels)
        true_labels.append(gold_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels
    )

    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [ ]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

model = AutoModelForTokenClassification.from_pretrained(
    "microsoft/deberta-v3-base",
    num_labels=11,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
training_args = TrainingArguments(
    output_dir="my_ner_model",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    push_to_hub=False,
    fp16=True,              
    dataloader_pin_memory=True
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()